# AIAP 24 Delivery Dataset: Exploratory Data Analysis

**Student role:** AI student preparing assessment EDA presentation.  
**Dataset:** `delivery.db`, loaded from the local `data` folder.  
**Tools:** SQLite, pandas, NumPy, and notebook-native HTML/SVG charts.

## Presentation objectives

- Understand delivery operations and customer feedback patterns.
- Identify data quality issues before modelling work.
- Explain service performance using interpretable statistics.
- Surface features likely useful for machine learning.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML(str):
        pass
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Required local dataset location.
DATA_PATH = Path("data") / "delivery.db"

# Optional fallback if assessor places notebook inside data/.
if not DATA_PATH.exists() and Path("delivery.db").exists():
    DATA_PATH = Path("delivery.db")

assert DATA_PATH.exists(), f"Database not found: {DATA_PATH}"
print(f"Using database: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size / 1_000_000:.2f} MB")

In [ ]:
def bar_chart(df, label_col, value_col, title, value_fmt="{:.1f}", color="#0070c0", width=780):
    data = df[[label_col, value_col]].dropna().copy()
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
    data = data.dropna()
    if data.empty:
        display(HTML(f"<b>{title}</b><br>No data available."))
        return
    max_val = data[value_col].abs().max()
    left, top, bar_h, gap = 190, 52, 26, 12
    chart_w = width - left - 110
    height = top + len(data) * (bar_h + gap) + 48
    rows = [f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">']
    rows.append(f'<text x="20" y="28" font-family="Arial" font-size="20" font-weight="700" fill="#101010">{title}</text>')
    for i, row in data.reset_index(drop=True).iterrows():
        label = str(row[label_col])
        value = float(row[value_col])
        y = top + i * (bar_h + gap)
        bar_w = 0 if max_val == 0 else chart_w * abs(value) / max_val
        rows.append(f'<text x="20" y="{y+18}" font-family="Arial" font-size="13" fill="#5f6f83">{label}</text>')
        rows.append(f'<rect x="{left}" y="{y}" width="{chart_w}" height="{bar_h}" fill="#e8eef5"/>')
        rows.append(f'<rect x="{left}" y="{y}" width="{bar_w}" height="{bar_h}" fill="{color}"/>')
        rows.append(f'<text x="{left + bar_w + 8}" y="{y+18}" font-family="Arial" font-size="12" fill="#101010">{value_fmt.format(value)}</text>')
    rows.append("</svg>")
    display(HTML("".join(rows)))


def line_chart(df, x_col, y_col, title, y_fmt="{:.0f}", color="#0070c0", width=780, height=360):
    data = df[[x_col, y_col]].dropna().copy()
    data[y_col] = pd.to_numeric(data[y_col], errors="coerce")
    data = data.dropna()
    if len(data) < 2:
        display(HTML(f"<b>{title}</b><br>Not enough data."))
        return
    left, right, top, bottom = 70, 30, 55, 55
    chart_w = width - left - right
    chart_h = height - top - bottom
    y_min, y_max = data[y_col].min(), data[y_col].max()
    if y_min == y_max:
        y_min, y_max = y_min - 1, y_max + 1
    pts = []
    for i, (_, row) in enumerate(data.reset_index(drop=True).iterrows()):
        x = left + chart_w * i / (len(data) - 1)
        y = top + chart_h * (1 - (row[y_col] - y_min) / (y_max - y_min))
        pts.append((x, y, row[x_col], row[y_col]))
    polyline = " ".join(f"{x:.1f},{y:.1f}" for x, y, _, _ in pts)
    rows = [f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">']
    rows.append(f'<text x="20" y="28" font-family="Arial" font-size="20" font-weight="700" fill="#101010">{title}</text>')
    rows.append(f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top+chart_h}" stroke="#cfd9e5"/>')
    rows.append(f'<line x1="{left}" y1="{top+chart_h}" x2="{left+chart_w}" y2="{top+chart_h}" stroke="#cfd9e5"/>')
    rows.append(f'<polyline points="{polyline}" fill="none" stroke="{color}" stroke-width="4"/>')
    for x, y, _, _ in pts:
        rows.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="4" fill="white" stroke="{color}" stroke-width="3"/>')
    for x, y, label, value in [pts[0], pts[-1]]:
        rows.append(f'<text x="{x-28:.1f}" y="{y-10:.1f}" font-family="Arial" font-size="12" fill="#101010">{y_fmt.format(value)}</text>')
        rows.append(f'<text x="{x-42:.1f}" y="{height-20}" font-family="Arial" font-size="12" fill="#5f6f83">{label}</text>')
    rows.append("</svg>")
    display(HTML("".join(rows)))


def metric_cards(metrics):
    cards = []
    for label, value, color in metrics:
        cards.append(
            f'<div style="border-left:8px solid {color}; background:white; padding:14px 16px; '
            f'box-shadow:0 1px 3px #d8e3ef; min-width:170px;">'
            f'<div style="font-family:Arial; font-size:26px; font-weight:700; color:{color};">{value}</div>'
            f'<div style="font-family:Arial; font-size:13px; color:#5f6f83;">{label}</div></div>'
        )
    display(HTML(f'<div style="display:flex; flex-wrap:wrap; gap:12px;">{"".join(cards)}</div>'))

## Step 1: Load database and inspect schema

**Steps taken**
- Connected SQLite using relative database path.
- Listed tables, columns, and row counts.

**Purpose**
- Confirm available data before analysis begins.
- Validate schema against assessment documentation.

**Conclusions**
- Database contains deliveries and feedback tables.
- Feedback table covers only some deliveries.

**Statistics interpretation**
- Row counts define analysis population size.
- Table columns reveal possible feature groups.
- Join keys determine valid relationship checks.

In [ ]:
with sqlite3.connect(DATA_PATH) as conn:
    tables = pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','view')", conn)
    schema = {table: pd.read_sql(f"PRAGMA table_info({table})", conn) for table in tables["name"]}
    counts = pd.DataFrame([
        {"table": table, "rows": pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]}
        for table in tables["name"]
    ])
display(tables)
display(counts)
for table, info in schema.items():
    print(f"\n{table}")
    display(info[["name", "type"]])

## Step 2: Read tables and standardise types

**Steps taken**
- Loaded SQL tables into pandas dataframes.
- Converted dates and numeric text columns.

**Purpose**
- Enable reliable arithmetic and grouping operations.
- Prevent text values distorting numeric statistics.

**Conclusions**
- Several numeric fields were stored as text.
- Datetime conversion enabled service-time features.

**Statistics interpretation**
- Coercion missingness signals dirty source records.
- Datetime ranges show operational coverage period.
- Numeric dtypes support modelling-ready transformations.

In [ ]:
with sqlite3.connect(DATA_PATH) as conn:
    deliveries = pd.read_sql("SELECT * FROM deliveries", conn)
    feedback = pd.read_sql("SELECT * FROM feedback", conn)

deliveries["branch_clean"] = deliveries["branch"].str.strip().str.title().replace({"Noth": "North", "Cnetral": "Central"})
deliveries["parcel_category_clean"] = deliveries["parcel_category"].str.strip().str.lower().replace({
    "over-sized": "oversized", "refrig.": "refrigerated"
})
deliveries["vehicle_type_clean"] = deliveries["vehicle_type"].str.strip().str.lower()
deliveries["payment_method_clean"] = deliveries["payment_method"].str.strip().str.lower()
deliveries["delivery_priority_clean"] = deliveries["delivery_priority"].str.strip().str.title()

for col in ["booking_datetime", "pickup_datetime", "promised_delivery_datetime", "delivery_datetime"]:
    deliveries[col] = pd.to_datetime(deliveries[col], errors="coerce")
feedback["feedback_datetime"] = pd.to_datetime(feedback["feedback_datetime"], errors="coerce")

for col in ["distance_km", "parcel_weight_kg", "parcel_value_sgd", "num_stops_on_route", "driver_experience_months"]:
    deliveries[col] = pd.to_numeric(deliveries[col], errors="coerce")
feedback["rating"] = pd.to_numeric(feedback["rating"], errors="coerce")

display(pd.DataFrame({
    "dataset": ["deliveries", "feedback"],
    "rows": [len(deliveries), len(feedback)],
    "columns": [deliveries.shape[1], feedback.shape[1]],
    "date_start": [deliveries["booking_datetime"].min(), feedback["feedback_datetime"].min()],
    "date_end": [deliveries["booking_datetime"].max(), feedback["feedback_datetime"].max()],
}))

## Step 3: Join feedback and engineer service features

**Steps taken**
- Left-joined feedback onto delivery records.
- Created timing, lateness, and feedback indicators.

**Purpose**
- Link operations with customer satisfaction outcomes.
- Derive interpretable service performance variables.

**Conclusions**
- Multiple feedback rows expand joined records slightly.
- Lateness becomes central satisfaction explanatory signal.

**Statistics interpretation**
- Feedback rate indicates observable satisfaction coverage.
- Median delivery hours describe typical operations.
- Lateness rate summarizes promise reliability risk.

In [ ]:
df = deliveries.merge(feedback, on="delivery_id", how="left", suffixes=("", "_feedback"))
df["pickup_wait_hours"] = (df["pickup_datetime"] - df["booking_datetime"]).dt.total_seconds() / 3600
df["promised_delivery_hours"] = (df["promised_delivery_datetime"] - df["booking_datetime"]).dt.total_seconds() / 3600
df["actual_delivery_hours"] = (df["delivery_datetime"] - df["booking_datetime"]).dt.total_seconds() / 3600
df["lateness_hours"] = (df["delivery_datetime"] - df["promised_delivery_datetime"]).dt.total_seconds() / 3600
df["is_late"] = df["lateness_hours"] > 0
df["has_feedback"] = df["rating"].notna()
df["booking_month"] = df["booking_datetime"].dt.to_period("M").astype(str)
df["booking_day_name"] = df["booking_datetime"].dt.day_name()

metric_cards([
    ("delivery rows", f"{len(deliveries):,}", "#0070c0"),
    ("feedback rows", f"{len(feedback):,}", "#00b050"),
    ("feedback coverage", f"{df['has_feedback'].mean():.1%}", "#ff2a00"),
    ("late delivery rate", f"{df['is_late'].mean():.1%}", "#0070c0"),
    ("median delivery hours", f"{df['actual_delivery_hours'].median():.1f}", "#00b050"),
])
display(pd.DataFrame({
    "check": ["duplicate delivery_id rows", "duplicate feedback_id rows", "deliveries with multiple feedback", "negative delivery durations"],
    "count": [deliveries["delivery_id"].duplicated().sum(), feedback["feedback_id"].duplicated().sum(), (feedback["delivery_id"].value_counts() > 1).sum(), (df["actual_delivery_hours"] < 0).sum()]
}))

## Step 4: Assess missingness and data quality

**Steps taken**
- Measured missing percentages across joined fields.
- Reviewed inconsistent categorical label variants.

**Purpose**
- Identify cleaning needs before modelling work.
- Avoid biased summaries from incomplete variables.

**Conclusions**
- Feedback missingness dominates the joined dataset.
- Category typos require standardisation before grouping.

**Statistics interpretation**
- High missing ratings limit satisfaction supervision.
- Low operational missingness supports feature engineering.
- Duplicate keys need careful aggregation decisions.

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False).head(14).reset_index()
missing.columns = ["field", "missing_percent"]
bar_chart(missing, "field", "missing_percent", "Top Missing Fields", "{:.1f}%", color="#ff2a00")

display(pd.DataFrame({
    "field": ["branch", "parcel_category", "vehicle_type"],
    "raw_values": [
        ", ".join(sorted(deliveries["branch"].dropna().unique())),
        ", ".join(sorted(deliveries["parcel_category"].dropna().unique())),
        ", ".join(sorted(deliveries["vehicle_type"].dropna().unique())),
    ],
    "clean_values": [
        ", ".join(sorted(deliveries["branch_clean"].dropna().unique())),
        ", ".join(sorted(deliveries["parcel_category_clean"].dropna().unique())),
        ", ".join(sorted(deliveries["vehicle_type_clean"].dropna().unique())),
    ],
}))

## Step 5: Explore delivery timing and SLA performance

**Steps taken**
- Summarised actual delivery and lateness distributions.
- Compared late versus on-time satisfaction outcomes.

**Purpose**
- Measure whether promised service is achieved.
- Link operational delay with customer experience.

**Conclusions**
- Most deliveries finish within one working day.
- Late deliveries receive substantially lower ratings.

**Statistics interpretation**
- Median reflects typical customer waiting time.
- Upper quantiles expose operational delay tails.
- Lateness separates satisfaction outcomes strongly.

In [ ]:
display(df[["pickup_wait_hours", "actual_delivery_hours", "promised_delivery_hours", "lateness_hours"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T)

late_rating = df.groupby("is_late").agg(
    deliveries=("delivery_id", "count"),
    feedback_rate=("rating", lambda s: s.notna().mean()),
    mean_rating=("rating", "mean"),
    median_rating=("rating", "median"),
    median_delivery_hours=("actual_delivery_hours", "median"),
).reset_index()
late_rating["status"] = late_rating["is_late"].map({False: "On time", True: "Late"})
display(late_rating[["status", "deliveries", "feedback_rate", "mean_rating", "median_rating", "median_delivery_hours"]])
bar_chart(late_rating, "status", "mean_rating", "Average Rating by SLA Status", "{:.2f}", color="#0070c0")

## Step 6: Analyse customer feedback distribution

**Steps taken**
- Counted rating values and feedback coverage.
- Checked comments and feedback timing availability.

**Purpose**
- Understand label balance for satisfaction modelling.
- Evaluate whether comments add qualitative signals.

**Conclusions**
- Ratings are skewed toward positive scores.
- Nonresponse may hide dissatisfied customer segments.

**Statistics interpretation**
- Mean rating shows overall satisfaction level.
- Median rating indicates dominant positive experience.
- Rating imbalance affects model evaluation strategy.

In [ ]:
rating_counts = feedback["rating"].value_counts(dropna=False).sort_index().reset_index()
rating_counts.columns = ["rating", "count"]
rating_counts["rating_label"] = rating_counts["rating"].astype("Int64").astype(str).replace("<NA>", "Missing")
metric_cards([
    ("mean rating", f"{feedback['rating'].mean():.2f}", "#0070c0"),
    ("median rating", f"{feedback['rating'].median():.0f}", "#00b050"),
    ("missing feedback ratings", f"{feedback['rating'].isna().sum():,}", "#ff2a00"),
])
bar_chart(rating_counts, "rating_label", "count", "Customer Rating Distribution", "{:.0f}", color="#00b050")
display(pd.DataFrame({
    "metric": ["feedback rows", "comments present", "comments missing", "comment coverage"],
    "value": [len(feedback), feedback["comment"].notna().sum(), feedback["comment"].isna().sum(), f"{feedback['comment'].notna().mean():.1%}"],
}))

## Step 7: Compare branches, priorities, and parcel categories

**Steps taken**
- Grouped performance by operational segments.
- Ranked segments using lateness and ratings.

**Purpose**
- Locate where service quality differs most.
- Identify controllable drivers for improvement.

**Conclusions**
- Priority tiers show clear service differentiation.
- Refrigerated and oversized parcels are riskier.

**Statistics interpretation**
- Group rates reveal operational heterogeneity.
- Segment medians reduce outlier distortion.
- Rating differences indicate customer-facing impact.

In [ ]:
def segment_summary(group_col):
    return (df.groupby(group_col, dropna=False)
            .agg(deliveries=("delivery_id", "count"),
                 late_rate=("is_late", "mean"),
                 median_delivery_hours=("actual_delivery_hours", "median"),
                 mean_rating=("rating", "mean"),
                 feedback_rate=("rating", lambda s: s.notna().mean()))
            .reset_index()
            .sort_values("late_rate", ascending=False))

branch_summary = segment_summary("branch_clean")
priority_summary = segment_summary("delivery_priority_clean")
category_summary = segment_summary("parcel_category_clean")
vehicle_summary = segment_summary("vehicle_type_clean")
for name, table in [("Branch summary", branch_summary), ("Priority summary", priority_summary), ("Parcel category summary", category_summary), ("Vehicle summary", vehicle_summary)]:
    print(name)
    display(table)
bar_chart(priority_summary, "delivery_priority_clean", "late_rate", "Late Rate by Delivery Priority", "{:.1%}", color="#ff2a00")
bar_chart(category_summary, "parcel_category_clean", "late_rate", "Late Rate by Parcel Category", "{:.1%}", color="#0070c0")

## Step 8: Study numeric relationships with ratings

**Steps taken**
- Correlated rating against operational numeric features.
- Compared direction and strength of relationships.

**Purpose**
- Identify likely predictive modelling features.
- Separate strong signals from weak variables.

**Conclusions**
- Lateness has strongest negative rating relationship.
- Driver experience shows positive satisfaction association.

**Statistics interpretation**
- Correlations measure linear directional association.
- Negative values imply lower expected ratings.
- Weak correlations may still support interactions.

In [ ]:
relationship_cols = ["rating", "actual_delivery_hours", "lateness_hours", "distance_km", "parcel_weight_kg", "parcel_value_sgd", "num_stops_on_route", "driver_experience_months"]
corr = df[relationship_cols].corr(numeric_only=True)["rating"].drop("rating").sort_values().reset_index()
corr.columns = ["feature", "correlation_with_rating"]
display(corr)
bar_chart(corr, "feature", "correlation_with_rating", "Correlation with Customer Rating", "{:.2f}", color="#0070c0")

df["lateness_bucket"] = pd.cut(df["lateness_hours"], bins=[-np.inf, -1, 0, 1, 3, np.inf], labels=["Early >1h", "Early 0-1h", "Late 0-1h", "Late 1-3h", "Late >3h"])
bucket_rating = df.groupby("lateness_bucket", observed=True).agg(deliveries=("delivery_id", "count"), mean_rating=("rating", "mean"), feedback_rate=("rating", lambda s: s.notna().mean())).reset_index()
display(bucket_rating)
bar_chart(bucket_rating, "lateness_bucket", "mean_rating", "Rating by Lateness Bucket", "{:.2f}", color="#ff2a00")

## Step 9: Review demand over time

**Steps taken**
- Aggregated bookings by month and weekday.
- Compared volume against late delivery patterns.

**Purpose**
- Detect seasonality and workload pressure periods.
- Support operational planning and capacity decisions.

**Conclusions**
- Monthly volume varies across observation window.
- Time features may improve predictive models.

**Statistics interpretation**
- Volume trends reveal changing demand exposure.
- Late rates normalize across unequal volumes.
- Calendar variables capture operational context.

In [ ]:
monthly = df.groupby("booking_month").agg(deliveries=("delivery_id", "count"), late_rate=("is_late", "mean"), mean_rating=("rating", "mean")).reset_index()
display(monthly)
line_chart(monthly, "booking_month", "deliveries", "Monthly Delivery Volume", "{:.0f}", color="#0070c0")
line_chart(monthly, "booking_month", "late_rate", "Monthly Late Delivery Rate", "{:.1%}", color="#ff2a00")

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = df.groupby("booking_day_name").agg(deliveries=("delivery_id", "count"), late_rate=("is_late", "mean")).reindex(weekday_order).reset_index()
display(weekday)
bar_chart(weekday, "booking_day_name", "deliveries", "Delivery Volume by Booking Weekday", "{:.0f}", color="#00b050")

## Step 10: Summarise implications for modelling

**Steps taken**
- Consolidated EDA findings into modelling guidance.
- Identified cleaning, feature, and target choices.

**Purpose**
- Translate exploration into machine learning readiness.
- Reduce modelling risk before pipeline development.

**Conclusions**
- SLA lateness should be core feature.
- Cleaning categories is required before encoding.

**Statistics interpretation**
- Missing labels constrain supervised training rows.
- Class imbalance requires careful validation metrics.
- Feature distributions guide transformations and clipping.

In [ ]:
vip_late = priority_summary.loc[priority_summary["delivery_priority_clean"] == "Vip", "late_rate"].iloc[0]
driver_corr = corr.loc[corr["feature"] == "driver_experience_months", "correlation_with_rating"].iloc[0]
eda_findings = pd.DataFrame({
    "finding": ["Feedback coverage is incomplete", "Late deliveries rate much lower", "Priority tiers separate service speed", "Parcel categories carry operational risk", "Category labels contain messy variants", "Driver experience relates positively"],
    "evidence": [f"Feedback coverage is {df['has_feedback'].mean():.1%}", f"Late mean rating {df.loc[df['is_late'], 'rating'].mean():.2f}", f"VIP late rate {vip_late:.1%}", f"Highest category late rate {category_summary['late_rate'].max():.1%}", "Raw labels include typos and casing variants", f"Correlation {driver_corr:.2f}"],
    "modelling_implication": ["Use observed ratings carefully", "Engineer lateness and delivery-time features", "Encode priority as ordered categorical feature", "Include category and weight interaction checks", "Standardise text categories before encoding", "Retain driver experience as candidate predictor"],
})
display(eda_findings)

# Final EDA Takeaways

- The delivery operation is mostly timely but uneven.
- Lateness is the clearest satisfaction risk factor.
- Feedback availability limits supervised modelling coverage.
- Priority, category, branch, and driver features matter.
- Cleaning labels and duplicates is mandatory first.
- Notebook uses lightweight dependencies for reproducibility.